<a href="https://colab.research.google.com/github/shikhar286/Agentic-AI-Portfolio-Shikhar-2026/blob/main/RAG_singlePDF_HR_Plociy_Q%26A_using_OPENAI/RAG_singlePDF_HR_Plociy_Q%26A_using_OPENAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu

import os
from google.colab import userdata, files
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Securely fetching the API key from Colab secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Step 1: File upload dialogue for Google Colab
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Step 2: Load PDF and split into manageable chunks locally
# Using RecursiveCharacterTextSplitter to maintain semantic context
loader = PyPDFLoader(file_name)
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(loader.load())

# Step 3: Create Vector Store using FAISS and OpenAI Embeddings
vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())

'''
Step 4: Persistence Layer
We save the FAISS index locally to './faiss_db'.
As long as the source PDF remains the same, we can reload this local DB
instead of re-embedding the entire document. This optimizes performance
and saves on OpenAI Embedding token costs.
Note: OpenAI API requires a minimum $5 credits as of April 2026.
'''
vectorstore.save_local('./faiss_db')

print(f'Local FAISS DB saved at: ./faiss_db')
print(f'Total vectors/chunks indexed: {vectorstore.index.ntotal}')

# Step 5: Initialize the Retriever object (The 'Search Agent')
retriever = vectorstore.as_retriever()

'''
Step 6: Prompt Engineering
Constructing a ChatPrompt that combines the User Query with retrieved Context.
By default, the retriever fetches the Top-K (k=4) most mathematically relevant
chunks from the vectorstore to provide as context to the LLM.
'''
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Use only the provided context to answer the question.\n\nContext:\n{context}'),
    ('user', '{question}')
])

# Step 7: The RAG Chain (LCEL - LangChain Expression Language)
# This pipeline links the Retriever, Prompt, LLM, and Output Parser.
chain = (
    {'context': retriever, 'question': lambda x: x}
    | prompt
    | ChatOpenAI(model='gpt-4o-mini', temperature=0)
    | StrOutputParser()
)

# Step 8: Execution
# Testing the chain with a specific policy-related query
print(chain.invoke('Help me to understand Notice Period Policy.'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


Saving sample_hr_policy.pdf to sample_hr_policy.pdf
Local FAISS DB saved at: ./faiss_db
Total vectors/chunks indexed: 27
The Notice Period Policy outlines the amount of notice an employee must give when resigning from their position, as well as the notice period required by the company when terminating an employee. Here are the key points:

### Employee Resignation Notice Period:
- **Non-management Employees**: 2 weeks
- **Management Employees**: 4 weeks
- **Executive Employees**: 8 weeks

### Company-Initiated Notice Period:
When the company initiates termination, the notice period varies based on the length of service:
- **Probationary Period**: 1 week
- **0-2 Years of Service**: 2 weeks
- **3-5 Years of Service**: 4 weeks
- **6+ Years of Service**: 6 weeks

### Additional Information:
- **Severance**: May be provided at the company's discretion based on circumstances and tenure, typically 1-2 weeks of pay for each year of service.
- **Return of Company Property**: Employees must ret